# MCP, Part 2 — Connecting a Client & an Agent
### Module 7 — Model Context Protocol (Part 2 of 2)

**Goal:** open a **client** connection to the `mcp_server.py` you met in Part 1, **discover and call** its tools, read its resource, and finally let an **LLM agent** drive those tools on its own — close to Omar Santos's `cyber_agent.py`.

> **Setup:** `pip install mcp langchain-mcp-adapters langgraph langchain-openai`. The client launches the server over **stdio**, so `mcp_server.py` and `cve_knowledge_base.md` must be in this folder. The agent step is guarded on `OPENAI_API_KEY`. Ships without outputs. Do Part 1 first.

## 🛠️ Setup — a Jupyter-safe way to run async MCP code

MCP clients are **async**, and Jupyter already runs an event loop, so we run each client call in a small worker thread with its own loop. We also send the server's stderr to a **real file** (`mcp_server_stderr.log`) — passing Jupyter's `sys.stderr` triggers a `fileno` error.

In [ ]:
# pip install mcp langchain-mcp-adapters langgraph langchain-openai python-dotenv
import os, sys, asyncio, threading
try:
    from dotenv import load_dotenv, find_dotenv; load_dotenv(find_dotenv())
except Exception:
    pass
HAS_KEY = bool(os.getenv("OPENAI_API_KEY"))

def run_async(coro):
    """Run a coroutine to completion in a fresh loop on a worker thread.
    Needed because Jupyter already owns the main event loop."""
    box = {}
    def worker():
        loop = asyncio.new_event_loop(); asyncio.set_event_loop(loop)
        try:
            box["value"] = loop.run_until_complete(coro)
        except Exception as e:
            box["error"] = e
        finally:
            loop.close()
    t = threading.Thread(target=worker); t.start(); t.join()
    if "error" in box: raise box["error"]
    return box["value"]

print("Setup ready. Key found:", HAS_KEY, "| Python:", sys.executable)

# 🔌 1 — Connect & Discover the Server's Tools

- `StdioServerParameters(command=python, args=['mcp_server.py'])`
- `stdio_client(...)` launches the server; `ClientSession` talks to it
- `session.initialize()` does the MCP handshake
- `session.list_tools()` discovers what the server offers

> ### 🎤 Instructor Script
>
> First we connect. We describe how to launch the server with StdioServerParameters — just run mcp_server.py with this Python. The stdio_client starts that subprocess, and a ClientSession wraps the connection. We call initialize to perform the MCP handshake, then list_tools to discover what the server exposes — the same port_scan, lookup_cve, and list_targets we built in Part 1. Notice the client didn't need to know those names in advance; it discovered them. That discovery is the heart of why MCP is reusable.

In [ ]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# How to start the server (client launches it over stdio):
server = StdioServerParameters(command=sys.executable, args=["mcp_server.py"])

async def discover():
    errlog = open("mcp_server_stderr.log", "w")          # real file (avoids Jupyter fileno error)
    async with stdio_client(server, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()                   # MCP handshake
            tools = await session.list_tools()
            return [(t.name, t.description.splitlines()[0]) for t in tools.tools]

for name, desc in run_async(discover()):
    print(f"- {name}: {desc}")

# 🛠️ 2 — Call the Tools & Read the Resource

- `session.call_tool(name, {args})` invokes a tool
- Result text is in `result.content[0].text`
- `session.read_resource('kb://cve')` fetches read-only data
- Same calls a human or an LLM agent would make

> ### 🎤 Instructor Script
>
> Now we actually use the server. call_tool runs a tool by name with a dictionary of arguments — we scan a lab host and look up a CVE for vsftpd — and the text comes back in the result's content. We also read the kb://cve resource, which is read-only data rather than an action. These are exactly the calls an agent will make in the next step; we're just doing them by hand first so you can see the request and response clearly.

In [ ]:
async def use_tools():
    errlog = open("mcp_server_stderr.log", "w")
    async with stdio_client(server, errlog=errlog) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            scan = await session.call_tool("port_scan", {"target": "10.0.0.5"})
            cve  = await session.call_tool("lookup_cve", {"service": "vsftpd 2.3.4"})
            kb   = await session.read_resource("kb://cve")
            return scan.content[0].text, cve.content[0].text, kb.contents[0].text

scan_out, cve_out, kb_out = run_async(use_tools())
print("PORT SCAN:\n", scan_out, "\n")
print("CVE LOOKUP:\n", cve_out, "\n")
print("RESOURCE kb://cve (first 200 chars):\n", kb_out[:200], "...")

# 🤖 3 — Let an LLM Agent Drive the Tools

- `langchain-mcp-adapters` turns MCP tools into LangChain tools
- `create_react_agent(llm, tools)` builds a tool-using agent
- The LLM reads each tool's description and decides what to call
- Close to Omar's `cyber_agent.py` — needs `OPENAI_API_KEY`

> ### 🎤 Instructor Script
>
> Finally the payoff: instead of us choosing tools, the LLM does. The langchain-mcp-adapters library connects to our server and converts its MCP tools into LangChain tools. We hand those to create_react_agent — the same LangGraph agent builder from last module — and ask a natural question. The model reads the tool descriptions, decides to scan then look up the CVE, and writes a report. That is an autonomous security agent reaching real tools through MCP, just like Omar's cyber_agent. It only runs if you have an OpenAI key; otherwise the manual calls above already showed the full capability.

In [ ]:
if HAS_KEY:
    from langchain_mcp_adapters.client import MultiServerMCPClient
    from langgraph.prebuilt import create_react_agent
    from langchain_openai import ChatOpenAI

    async def run_agent(question):
        client = MultiServerMCPClient({
            "cyber": {"command": sys.executable, "args": ["mcp_server.py"], "transport": "stdio"}
        })
        tools = await client.get_tools()                       # MCP tools -> LangChain tools
        agent = create_react_agent(ChatOpenAI(model="gpt-4o-mini", temperature=0), tools)
        result = await agent.ainvoke({"messages": [("user", question)]})
        return result["messages"][-1].content

    answer = run_async(run_agent(
        "Scan host 10.0.0.5, then look up any CVE for the services you find, and summarize the risk."))
    print(answer)
else:
    print("No OPENAI_API_KEY — skipping the live agent. The manual tool calls above show the same tools.")

# 🛡️ 4 — Safety & Ethics

- Our tools are **simulated** — `port_scan` sends no packets
- Real scans/exploits: **only** on systems you're authorized to test
- Keep API keys in `.env` / Colab Secrets — never in code
- Treat tool outputs as untrusted input to the model (prompt-injection risk)

> ### 🎤 Instructor Script
>
> A safety word before the project. Everything here is simulated — the port scan returns canned data and sends no packets. When you wire MCP tools to real scanners or exploits, run them only against systems you are explicitly authorized to test, such as your isolated Metasploitable lab or an approved practice target. Keep keys in a dotenv file, never in code. And remember that anything a tool returns becomes input to the model, so a malicious server or web page could try to manipulate your agent — a real concern we revisit in the orchestration module.

# ✅ Summary — Part 2 & the Module

- A **client** launches the server over stdio and discovers its tools
- `call_tool` / `read_resource` use the server's capabilities
- `langchain-mcp-adapters` + `create_react_agent` → an autonomous agent
- You connected an agent to MCP tools, safely and reusably
- → Mini-Project 5: build your own MCP server **and** client

> ### 🎤 Instructor Script
>
> To wrap up MCP: in Part 1 you built a server; in Part 2 you connected a client that discovered and called its tools and read its resource, then let an LLM agent drive those tools on its own. That client-server split is what makes the tools reusable across any MCP host. Mini-Project 5 asks you to build both sides — your own server exposing a security tool, and a client (or agent) that uses it. Next module we orchestrate agents like this against authorized targets.

In [ ]:
print("Part 2 done: client, tool calls, resource, and an MCP-powered agent.")
print("Next: mcp_sample.ipynb (template) and Mini-Project 5.")